In [ ]:
import matplotlib.pyplot as plt
import sys
from pathlib import Path
ROOT = Path("..").resolve()          # absolute path to the project root
sys.path.append(str(ROOT))           # so that we can import from src
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)         # create a folder for figures, if it doesn't exist yet

from src.data import load_train

X, y = load_train()                      # X: (42000, 784), y: (42000,)

fig, axes = plt.subplots(2, 5, figsize=(8, 4))   # a 2-by-5 grid of empty plots
for i, ax in enumerate(axes.flat):               # axes.flat walks the grid one plot at a time
    ax.imshow(X[i].reshape(28, 28), cmap="gray") # one row of 784 -> a 28x28 picture
    ax.set_title(str(y[i]))                      # its label, as the title
    ax.axis("off")                               # hide tick marks
fig.tight_layout()                               # stop titles overlapping
fig.savefig(FIG_DIR / "sample_digits.png", dpi=150)
plt.show()

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import json
import numpy as np

ROOT = Path("..").resolve()
RUN_DIR = ROOT / "runs"
run_paths = list(RUN_DIR.glob("*_mlp"))
runs = {}

for path in run_paths:
    config_path = path / "config.json"
    with open(config_path, "r") as cf:
        config = json.load(cf)
    if config["commit"] == "1080f1d":
        history_path = path / "history.json"
        with open(history_path, "r") as his:
            history = json.load(his)
        runs[config["lr"]] = {"val": history["val_acc"], "loss": history["train_loss"]}

lrs = sorted(runs)
colors = plt.cm.Blues(np.linspace(0.45, 1.0, len(lrs)))

acc_fig, acc_axes = plt.subplots(2, 3, figsize=(11,6), sharex=True, sharey=True)

for acc_ax, lr in zip(acc_axes.flat, lrs):
    epochs = np.arange(1, len(runs[lr]["val"]) + 1)
    for other in lrs:
        acc_ax.plot(epochs, runs[other]["val"], color="lightgrey", linewidth=1)
    acc_ax.plot(epochs, runs[lr]["val"], color="tab:blue", linewidth=2)
    acc_ax.set_title(f"lr: {lr}")
    acc_ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    acc_ax.grid(alpha=0.3)
    acc_ax.spines[["top", "right"]].set_visible(False)

acc_fig.supxlabel("Epoch")
acc_fig.supylabel("Validation accuracy")
acc_fig.suptitle("Above lr=0.3, training collapses mid-run and recovers; final accuracies near-tie")
acc_fig.tight_layout()

loss_fig, loss_ax = plt.subplots()
for lr, color in zip(lrs, colors):
    epochs = np.arange(1, len(runs[lr]["loss"]) + 1)
    loss_ax.plot(epochs, runs[lr]["loss"], color=color, linewidth=2, label=f"lr = {lr}")
    loss_ax.annotate(f"lr {lr}", xy=(epochs[-1], runs[lr]["loss"][-1]),
                    xytext=(6, 0), textcoords="offset points", va="center", fontsize=9)

loss_ax.set(xlabel="Epoch", ylabel="Training loss (log scale)", yscale="log")
loss_ax.xaxis.set_major_locator(MaxNLocator(integer=True))
loss_ax.grid(alpha=0.3)
loss_ax.spines[["top", "right"]].set_visible(False)
loss_ax.legend(frameon=False)
loss_fig.suptitle("Higher learning rates lower loss faster; at 0.5 and 1.0 it spikes above ln 10")
loss_fig.tight_layout()